In [21]:
import random
import csv
import time
from collections import deque
import heapq
import pandas as pd
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq()

def call_gpt_oss_20b(maze):
    rows = len(maze)
    cols = len(maze[0])
    start, goal = find_points(maze)
    
    maze_str = ""
    for r in range(rows):
        row_str = ""
        for c in range(cols):
            row_str += f"{maze[r][c]} "
        maze_str += row_str.strip() + "\n"

    # Following the user's suggestion for multi-turn messages and specific parameters
    messages = [
        {
            "role": "user",
            "content": f"Here is the maze\n{maze_str}\nYou are a pathfinding agent. Solve this {rows}x{cols} maze.\n\nGOAL: Move from 'S' to 'G'.\nRULES:\n- 'S' (Start) is at {start}\n- 'G' (Goal) is at {goal}\n- '0' are open cells.\n- '1' are walls. Do not walk into walls.\n- Output ONLY the sequence of moves (UP, DOWN, LEFT, RIGHT).\n- Think step by step about your current coordinates.\n\nINSTRUCTIONS:\nFind the path from {start} to {goal}.\nList EACH move on a new line with current coordinates.\nExample:\n(0,0) -> RIGHT -> (0,1)\n(0,1) -> DOWN -> (1,1)\n\nFinal list of moves: MOVE, MOVE, MOVE...\nReturn ONLY a list of moves at the end."
        },
        {
            "role": "assistant",
            "content": "I’m missing the start location. Could you provide the coordinates for **S** (row, column) in the same format as the goal? Once I know the start, I can calculate and list the moves."
        },
        {
            "role": "user",
            "content": f"S {' '.join(['1' if c == 'G' else str(c) for c in maze[0][1:]])}\n{maze_str[maze_str.find('\n')+1:]}\n\nYou are a pathfinding agent. Solve this {rows}x{cols} maze.\n\nGOAL: Move from 'S' to 'G'. RULES:\n\n    'S' (Start) is at {start}\n    'G' (Goal) is at {goal}\n    '0' are open cells.\n    '1' are walls. Do not walk into walls.\n    Output ONLY the sequence of moves (UP, DOWN, LEFT, RIGHT).\n    Think step by step about your current coordinates.\n\nINSTRUCTIONS: Find the path from {start} to {goal}. List EACH move on a new line with current coordinates. Example: (0,0) -> RIGHT -> (0,1) (0,1) -> DOWN -> (1,1)\n\nFinal list of moves: MOVE, MOVE, MOVE... Return ONLY a list of moves at the end.\n"
        }
    ]

    try:
        completion = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            temperature=1,
            max_completion_tokens=8192,
            top_p=1,
            reasoning_effort="medium",
            stream=True,
            stop=None
        )
        
        full_response = ""
        for chunk in completion:
            full_response += chunk.choices[0].delta.content or ""
        return full_response.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

def parse_moves(text):
    text_upper = text.upper()
    
    # Handle both common variants of the final line
    if "FINAL LIST OF MOVES:" in text_upper:
        moves_part = text_upper.split("FINAL LIST OF MOVES:")[1]
    elif "FINAL LIST OF MOVES" in text_upper:
         moves_part = text_upper.split("FINAL LIST OF MOVES")[1]
    else:
        moves_part = text_upper

    # Clean the text and extract moves
    clean_text = moves_part.replace("\n", ",").replace(".", ",").replace(" ", "")
    parts = clean_text.split(",")
    valid = {"UP", "DOWN", "LEFT", "RIGHT"}
    return [m for m in parts if m in valid]

def moves_to_path(maze, moves):
    start, goal = find_points(maze)
    x, y = start
    path = [(x, y)]

    move_map = {
        "UP": (-1, 0),
        "DOWN": (1, 0),
        "LEFT": (0, -1),
        "RIGHT": (0, 1)
    }

    if not moves:
        return None

    for move in moves:
        dx, dy = move_map[move]
        nx, ny = x + dx, y + dy

        if not (0 <= nx < len(maze) and 0 <= ny < len(maze[0])):
            print(f"LLM Move {move} went out of bounds to ({nx}, {ny})")
            return None
        # проверка стен
        if maze[nx][ny] == 1:
            print(f"LLM Move {move} hit a wall at ({nx}, {ny})")
            return None

        x, y = nx, ny
        path.append((x, y))

        if (x, y) == goal:
            return path

    print(f"LLM finished moves but did not reach goal. Current: ({x}, {y}), Goal: {goal}")
    return None

def draw_llm_path(maze, path):
    maze_copy = [row[:] for row in maze]

    if path:
        for x, y in path:
            if maze_copy[x][y] not in ['S', 'G']:
                maze_copy[x][y] = 'L'

    return maze_copy

In [22]:
def generate_maze(size=11, seed=42):
    if size % 2 == 0:
        size += 1
    random.seed(seed)
    maze = [[1]*size for _ in range(size)]

    def carve(x, y):
        dirs = [(2,0), (-2,0), (0,2), (0,-2)]
        random.shuffle(dirs)
        for dx, dy in dirs:
            nx, ny = x+dx, y+dy
            if 0 < nx < size-1 and 0 < ny < size-1 and maze[nx][ny] == 1:
                maze[nx][ny] = 0
                maze[x+dx//2][y+dy//2] = 0
                carve(nx, ny)

    maze[1][1] = 0
    carve(1,1)

    maze[0][1] = 0
    maze[0][0] = 'S'
    maze[size-1][size-2] = 0
    maze[size-1][size-1] = 'G'

    # Add cycles by removing some random walls
    for _ in range(size):
        rx, ry = random.randint(1, size-2), random.randint(1, size-2)
        if maze[rx][ry] == 1:
            maze[rx][ry] = 0

    return maze

In [23]:
def print_maze(maze):
    for row in maze:
        print(" ".join(str(x) for x in row))

In [24]:
def find_points(maze):
    start = goal = None
    for i in range(len(maze)):
        for j in range(len(maze[0])):
            if maze[i][j] == 'S':
                start = (i, j)
            if maze[i][j] == 'G':
                goal = (i, j)
    return start, goal


def get_neighbors(pos, maze):
    x, y = pos
    directions = [(1,0), (-1,0), (0,1), (0,-1)]
    neighbors = []

    for dx, dy in directions:
        nx, ny = x+dx, y+dy
        if 0 <= nx < len(maze) and 0 <= ny < len(maze[0]):
            if maze[nx][ny] != 1:
                neighbors.append((nx, ny))

    return neighbors

In [25]:
def bfs(maze):
    start, goal = find_points(maze)
    queue = deque([(start, [start])])
    visited = set([start])

    while queue:
        current, path = queue.popleft()

        if current == goal:
            return path

        for neighbor in get_neighbors(current, maze):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None

In [26]:
def dfs(maze):
    start, goal = find_points(maze)
    stack = [(start, [start])]
    visited = set()

    while stack:
        current, path = stack.pop()

        if current == goal:
            return path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in get_neighbors(current, maze):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return None

In [27]:
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(maze):
    start, goal = find_points(maze)
    heap = [(0, start, [start])]
    visited = set()

    while heap:
        cost, current, path = heapq.heappop(heap)

        if current == goal:
            return path

        if current in visited:
            continue
        visited.add(current)

        for neighbor in get_neighbors(current, maze):
            new_cost = len(path)
            priority = new_cost + heuristic(neighbor, goal)
            heapq.heappush(heap, (priority, neighbor, path + [neighbor]))

    return None

In [28]:
def run_experiment(num_mazes=10, size=11):
    results = []

    for i in range(num_mazes):
        maze = generate_maze(size=size, seed=i)

        # BFS
        start_time = time.time()
        bfs_path = bfs(maze)
        bfs_time = time.time() - start_time

        # DFS
        start_time = time.time()
        dfs_path = dfs(maze)
        dfs_time = time.time() - start_time

        # A*
        start_time = time.time()
        astar_path = astar(maze)
        astar_time = time.time() - start_time

        # LLM
        start_time_llm = time.time()
        try:
            llm_output = call_gpt_oss_20b(maze)
            llm_time = time.time() - start_time_llm
            moves = parse_moves(llm_output)
            llm_path = moves_to_path(maze, moves)
        except Exception as e:
            llm_time = time.time() - start_time_llm
            print(f"LLM Error on maze {i}: {e}")
            llm_path = None

        results.append({
            "maze_id": i,
            "bfs_length": len(bfs_path) if bfs_path else 0,
            "dfs_length": len(dfs_path) if dfs_path else 0,
            "astar_length": len(astar_path) if astar_path else 0,
            "llm_length": len(llm_path) if llm_path else 0,
            "llm_success": llm_path is not None,
            "bfs_time": bfs_time,
            "dfs_time": dfs_time,
            "astar_time": astar_time,
            "llm_time": llm_time
        })
        time.sleep(1) # To avoid rate limits

    return results

In [29]:
df = pd.DataFrame(run_experiment(num_mazes=5, size=11))

df.to_csv("results_gpt-oss-20b.csv", index=False)

print(df.head())

LLM Move DOWN hit a wall at (8, 5)
LLM Move RIGHT hit a wall at (8, 2)
   maze_id  bfs_length  dfs_length  astar_length  llm_length  llm_success  \
0        0          29          31            29           0        False   
1        1          21          23            21          21         True   
2        2          21          35            21          25         True   
3        3          21          21            21          21         True   
4        4          29          37            29           0        False   

   bfs_time  dfs_time  astar_time   llm_time  
0  0.000154  0.000148    0.000160   6.771946  
1  0.000363  0.000306    0.000221   3.403545  
2  0.000352  0.000292    0.000470   6.612377  
3  0.000196  0.000156    0.000252  29.927891  
4  0.000293  0.000268    0.000463  17.753564  


In [30]:
maze = generate_maze(seed=0)

print("Maze:")
print_maze(maze)

# BFS for reference
bfs_path = bfs(maze)
astar_path = astar(maze)

llm_output = call_gpt_oss_20b(maze)
print("\nLLM raw output:", llm_output)

moves = parse_moves(llm_output)
llm_path = moves_to_path(maze, moves)

print("\nParsed moves:", moves)
print("LLM Path length:", len(llm_path) if llm_path else "Invalid path")
print("BFS Path length:", len(bfs_path) if bfs_path else 0)

if llm_path:
    print("\nMaze with LLM path:")
    print_maze(draw_llm_path(maze, llm_path))


Maze:
S 0 1 1 1 1 1 1 1 1 1
1 0 0 0 1 0 0 0 0 0 1
1 1 0 0 1 0 1 0 1 0 1
1 0 1 0 1 0 1 0 1 0 1
1 0 1 0 1 1 1 0 1 0 1
1 0 1 0 1 0 0 0 0 0 1
1 0 1 0 1 0 1 1 1 0 1
1 0 1 0 0 0 1 0 0 0 1
1 0 1 1 1 1 0 0 1 1 1
1 0 0 0 0 0 0 0 0 0 1
1 1 1 1 1 1 1 1 1 0 G

LLM raw output: 

Parsed moves: []
LLM Path length: Invalid path
BFS Path length: 29
